In this notebook, I verify the desired properties of tje extended transformer model I generate

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

import torch

from extend_model import generate_extended_tok_and_model, embed_method

In [8]:
MODEL_NAME = "Qwen/Qwen3.5-9B"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(model.get_output_embeddings().weight.shape)
print(len(tokenizer))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

torch.Size([248320, 4096])
248077


In [ ]:
# Modify model
model, extended_tokenizer = generate_extended_tok_and_model(
    model=model, 
    tokenizer=tokenizer, 
    emb_method=embed_method.PRIOR_REPRESENTATION_EMBED,
    data_path="./results/phrase_means.pt"
)

In [3]:
vocab_size = len(tokenizer)

# Test 1: no change to encoding

input = "blackmail"
print(tokenizer.encode(input))
print(extended_tokenizer.encode(input))

# Test 2: no change to decoding for general token

random_id = random.randint(0, len(tokenizer) - 1)

print(tokenizer.decode([random_id]))
print(extended_tokenizer.decode([random_id]))

# Test 3: able to decode new tokens

new_id = len(tokenizer)

print(extended_tokenizer.decode([new_id]))

[11124, 3585]
[11124, 3585]
 mh
 mh
New Hampshire


In [4]:
model.eval()

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids)

logits = out.logits  # [batch, seq_len, vocab_size]
print(logits.shape)

torch.Size([1, 5, 248330])


In [5]:
print(len(tokenizer))
print(len(extended_tokenizer))

248077
248087


In [6]:
print(model.get_output_embeddings().weight.shape)

torch.Size([248330, 4096])


In [10]:
print(model.get_output_embeddings().weight[248315])

tensor([-0.0048,  0.0016,  0.0005,  ..., -0.0077,  0.0020, -0.0018],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)
